# Project Setup

In [8]:
# Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [9]:
import torch

# Select GPU if available, otherwise use CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


In [10]:
# Import Path
from pathlib import Path

# Dataset path
DATA = Path(
    "/kaggle/input/datasets/francismon/curated-covid19-chest-xray-dataset/dataset"
)

# Check dataset
print("Dataset loaded successfully!")
print("Dataset path:", DATA)
print("Exists:", DATA.exists())

Dataset loaded successfully!
Dataset path: /kaggle/input/datasets/francismon/curated-covid19-chest-xray-dataset/dataset
Exists: True


# Dataset Understanding & Audit

In [11]:
# Inspect dataset structure

for folder in DATA.iterdir():
    if folder.is_dir():
        print(f"\n{folder.name}/")

        for class_folder in folder.iterdir():
            if class_folder.is_dir():
                count = len(list(class_folder.glob("*.jpg")))
                print(f"  └── {class_folder.name}/ : {count} images")


validation/
  └── 0_Normal/ : 654 images
  └── 1_Covid19/ : 256 images
  └── 2_Pneumonia/ : 931 images

train/
  └── 0_Normal/ : 2616 images
  └── 1_Covid19/ : 1025 images
  └── 2_Pneumonia/ : 3726 images


In [12]:
# Verify total and class counts

classes = ["0_Normal", "1_Covid19", "2_Pneumonia"]

total = 0
classes_count = {}
for c in classes:
    count = len(list(DATA.rglob(f"{c}/*.jpg")))
    classes_count.update({c: count})
    total += count
    print(f"{c}: {count}")

print("\nTotal images:", total)

0_Normal: 3270
1_Covid19: 1281
2_Pneumonia: 4657

Total images: 9208


In [13]:
# Verify train/validation counts

for split in ["train", "validation"]:
    print(f"\n{split.upper()}")

    total = 0

    for c in classes:
        count = len(list((DATA / split / c).glob("*.jpg")))
        total += count
        print(f"{c}: {count}")

    print("Total:", total)


TRAIN
0_Normal: 2616
1_Covid19: 1025
2_Pneumonia: 3726
Total: 7367

VALIDATION
0_Normal: 654
1_Covid19: 256
2_Pneumonia: 931
Total: 1841


In [14]:
from PIL import Image

files = list(DATA.rglob("*.jpg"))

chunk_size = 500

for start in range(0, len(files), chunk_size):

    chunk = files[start:start + chunk_size]

    dimensions = set()
    formats = set()
    modes = set()

    for f in chunk:
        with Image.open(f) as img:
            dimensions.add(img.size)
            formats.add(img.format)
            modes.add(img.mode)

    print(f"Chunk {start//chunk_size + 1}")
    print("Dimensions:", dimensions)
    print("Formats:", formats)
    print("Color modes:", modes)

Chunk 1
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 2
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 3
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 4
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 5
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 6
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 7
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 8
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 9
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 10
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 11
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 12
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 13
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color modes: {'RGB'}
Chunk 14
Dimensions: {(299, 299)}
Formats: {'JPEG'}
Color mo

In [15]:
# Inspect filename structure

for split in ["train", "validation"]:
    print(f"\n{split.upper()}")

    for c in classes:
        files = list((DATA / split / c).glob("*.jpg"))

        print(f"\n{c}:")
        for f in files[:5]:
            print(" ", f.name)


TRAIN

0_Normal:
  Normal (1525).jpg
  Normal (1926).jpg
  Normal (398).jpg
  Normal (1239).jpg
  Normal (2106).jpg

1_Covid19:
  COVID-19 (597).jpg
  COVID-19 (573).jpg
  COVID-19 (394).jpg
  COVID-19 (695).jpg
  COVID-19 (956).jpg

2_Pneumonia:
  Pneumonia-Bacterial (859).jpg
  Pneumonia-Viral (414).jpg
  Pneumonia-Bacterial (1013).jpg
  Pneumonia-Bacterial (283).jpg
  Pneumonia-Bacterial (999).jpg

VALIDATION

0_Normal:
  Normal (2814).jpg
  Normal (3095).jpg
  Normal (3186).jpg
  Normal (2823).jpg
  Normal (2888).jpg

1_Covid19:
  COVID-19 (1273).jpg
  COVID-19 (1188).jpg
  COVID-19 (1181).jpg
  COVID-19 (1240).jpg
  COVID-19 (1264).jpg

2_Pneumonia:
  Pneumonia-Viral (1593).jpg
  Pneumonia-Viral (1419).jpg
  Pneumonia-Viral (973).jpg
  Pneumonia-Viral (1570).jpg
  Pneumonia-Viral (733).jpg


In [16]:
# Check class imbalance

for name, count in classes_count.items():
    percentage = count / total * 100
    print(f"{name}: {count} ({percentage:.2f}%)")

0_Normal: 3270 (177.62%)
1_Covid19: 1281 (69.58%)
2_Pneumonia: 4657 (252.96%)


In [17]:
# Inspect Pneumonia filename subcategories
from collections import Counter
pneumonia_files = list(DATA.rglob("2_Pneumonia/*.jpg"))

subcategories = Counter()

for f in pneumonia_files:
    name = f.name.lower()

    if "bacterial" in name:
        subcategories["Bacterial"] += 1
    elif "viral" in name:
        subcategories["Viral"] += 1
    else:
        subcategories["Unknown"] += 1

print(subcategories)

Counter({'Bacterial': 3001, 'Viral': 1656})


# Research Dataset Split

In [18]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# Dataset root
#google drive 
# DATA_ROOT = Path(
#     "/content/drive/MyDrive/Colab Notebooks/"
#     "Curated COVID-19 Chest X-Ray Dataset"
# )

# # Output location
# OUTPUT = Path(
#     "/content/drive/MyDrive/Colab Notebooks/"
#     "covid_chest_x-ray_curated_dataset/data_split.csv"
# )


# for kaggle

DATA_ROOT = DATA

OUTPUT = Path(
    "/kaggle/working/data_split.csv"
)



# Classes
classes = {
    "0_Normal": 0,
    "1_Covid19": 1,
    "2_Pneumonia": 2
}

# --------------------------------------------------
# 1. Collect images using RELATIVE paths
# --------------------------------------------------

rows = []

for class_name, label in classes.items():
    for image_path in DATA_ROOT.rglob(f"{class_name}/*.jpg"):

        relative_path = image_path.relative_to(DATA_ROOT)

        rows.append({
            "path": str(relative_path),
            "label": label,
            "class": class_name
        })

df = pd.DataFrame(rows)

print("Total images:", len(df))

Total images: 9208


In [19]:
print(df.columns.tolist())
print(df.head())
print(df.shape)

['path', 'label', 'class']
                                    path  label     class
0  validation/0_Normal/Normal (2814).jpg      0  0_Normal
1  validation/0_Normal/Normal (3095).jpg      0  0_Normal
2  validation/0_Normal/Normal (3186).jpg      0  0_Normal
3  validation/0_Normal/Normal (2823).jpg      0  0_Normal
4  validation/0_Normal/Normal (2888).jpg      0  0_Normal
(9208, 3)


In [20]:
# 2. Create stratified 70/15/15 split
# --------------------------------------------------

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

# Add split column
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

# Combine
split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

In [21]:
print("=== SPLIT SIZES ===")
print(split_df["split"].value_counts())

print("\n=== CLASS DISTRIBUTION ===")
print(
    pd.crosstab(
        split_df["split"],
        split_df["class"]
    )
)

print("\n=== CLASS PERCENTAGES ===")
print(
    (
        pd.crosstab(
            split_df["split"],
            split_df["class"],
            normalize="index"
        ) * 100
    ).round(2)
)

print("\n=== DUPLICATE CHECK ===")
print("Duplicate paths:", split_df["path"].duplicated().sum())

print("\n=== TOTAL CHECK ===")
print("Total:", len(split_df))
print("Expected:", len(df))
print("Correct:", len(split_df) == len(df))

print("\n=== SPLIT CHECK ===")
print("Splits:", sorted(split_df["split"].unique()))

print("\n=== CLASS CHECK ===")
print("Classes:", sorted(split_df["class"].unique()))

=== SPLIT SIZES ===
split
train         6445
test          1382
validation    1381
Name: count, dtype: int64

=== CLASS DISTRIBUTION ===
class       0_Normal  1_Covid19  2_Pneumonia
split                                       
test             491        192          699
train           2289        897         3259
validation       490        192          699

=== CLASS PERCENTAGES ===
class       0_Normal  1_Covid19  2_Pneumonia
split                                       
test           35.53      13.89        50.58
train          35.52      13.92        50.57
validation     35.48      13.90        50.62

=== DUPLICATE CHECK ===
Duplicate paths: 0

=== TOTAL CHECK ===
Total: 9208
Expected: 9208
Correct: True

=== SPLIT CHECK ===
Splits: ['test', 'train', 'validation']

=== CLASS CHECK ===
Classes: ['0_Normal', '1_Covid19', '2_Pneumonia']


In [22]:
missing = [
    path for path in split_df["path"]
    if not (DATA_ROOT / path).exists()
]

print("\n=== FILE CHECK ===")
print("Missing files:", len(missing))

if missing:
    print("\nFirst missing files:")
    for path in missing[:10]:
        print(path)


=== FILE CHECK ===
Missing files: 0


In [23]:
# --------------------------------------------------
# 3. Save FINAL portable CSV
# --------------------------------------------------

split_df.to_csv(OUTPUT, index=False)

print("\nSaved:", OUTPUT)
print("\nSplit distribution:")
print(split_df["split"].value_counts())

print("\nClass distribution:")
print(
    split_df.groupby(["split", "class"])
    .size()
)

print("\nFirst 5 rows:")
display(split_df.head())


Saved: /kaggle/working/data_split.csv

Split distribution:
split
train         6445
test          1382
validation    1381
Name: count, dtype: int64

Class distribution:
split       class      
test        0_Normal        491
            1_Covid19       192
            2_Pneumonia     699
train       0_Normal       2289
            1_Covid19       897
            2_Pneumonia    3259
validation  0_Normal        490
            1_Covid19       192
            2_Pneumonia     699
dtype: int64

First 5 rows:


,path,label,class,split
0,train/2_Pneumonia/Pneumonia-Bacterial (1153).jpg,2,2_Pneumonia,train
1,train/0_Normal/Normal (1926).jpg,0,0_Normal,train
2,train/2_Pneumonia/Pneumonia-Viral (364).jpg,2,2_Pneumonia,train
3,train/2_Pneumonia/Pneumonia-Bacterial (973).jpg,2,2_Pneumonia,train
4,train/2_Pneumonia/Pneumonia-Bacterial (280).jpg,2,2_Pneumonia,train
